### 1. Hollys

In [ ]:
from bs4 import BeautifulSoup
import requests

URL='https://www.hollys.co.kr/store/korea/korStore2.do'
headers={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0'
}

In [ ]:
# 함수 설정 fetch, parse

def fetch(page):
    params={
        'pageNo':page,
        'sido':'',
        'gugun':'',
        'store':''
    }
    res=requests.get(URL,params=params,headers=headers,timeout=10) # timeout이 있어야 루프가 없다고 했지
    print(f'접속 상태:{res.status_code}')
    return res.text

def get_text(tag) -> str:
    return tag.text.strip() if tag else ''

def parse(html):
    soup=BeautifulSoup(html,'html.parser')
    trs=soup.select('table.tb_store tbody tr')
    rows=[]
    for tr in trs:
        td=tr.select('td')
        services=[img.get('alt','') for img in td[4].select('img')] # td의 5번째열 img에 해당하는 alt값
        rows.append({
            '지역':get_text(td[0]),
            '매장명':get_text(td[1]),
            '현황':get_text(td[2]),
            '주소':get_text(td[3]),
            '매장 서비스':services,
            '전화번호':get_text(td[5]),
        })
    return rows

parse(fetch(10))

접속 상태:200


[{'지역': '경기 화성시',
  '매장명': '화성만세DT점',
  '현황': '영업중',
  '주소': '경기도 화성시 향남읍 서해로 191 1층',
  '매장 서비스': ['DT 매장', '테라스', '주차'],
  '전화번호': '070-5159-5942'},
 {'지역': '대구 달서구',
  '매장명': '대구장기점',
  '현황': '영업중',
  '주소': '대구광역시 달서구 장기로 268 1~2층',
  '매장 서비스': ['흡연시설', '주차'],
  '전화번호': '053-522-0666'},
 {'지역': '부산 연제구',
  '매장명': '부산연산점',
  '현황': '영업중',
  '주소': '부산광역시 연제구 반송로 89 (연산동) 1층',
  '매장 서비스': ['주차'],
  '전화번호': '051-989-3296'},
 {'지역': '경기 안양시 동안구',
  '매장명': '평촌점',
  '현황': '영업중',
  '주소': '경기도 안양시 동안구 관평로170번길 43 (훼미리타운 125호) 1층',
  '매장 서비스': ['테라스', '흡연시설', '주차'],
  '전화번호': '031-385-3400'},
 {'지역': '인천 남동구',
  '매장명': '남동케이원점',
  '현황': '영업중',
  '주소': '인천광역시 남동구 비류대로 622 남동산단케이원 1층 105,106,108호',
  '매장 서비스': ['주차'],
  '전화번호': '032-821-9110'},
 {'지역': '충북 청주시 흥덕구',
  '매장명': '오송역사점',
  '현황': '영업중',
  '주소': '충청북도 청주시 흥덕구 오송읍 오송가락로 123 (오송역 2층 맞이방) 봉산리 370-31',
  '매장 서비스': ['주차'],
  '전화번호': '.'},
 {'지역': '경북 구미시',
  '매장명': '구미CGV점',
  '현황': '영업중',
  '주소': '경상북도 구미시 구미중앙로 44 (멀티복합상가) 원평동68-1, 1층',


In [ ]:
result=[]
for i in range(1,11): # i:페이지
    rows=parse(fetch(i))
    result.extend(rows)
    print(f'{i}페이지 확인')

접속 상태:200
1페이지 확인
접속 상태:200
2페이지 확인
접속 상태:200
3페이지 확인
접속 상태:200
4페이지 확인
접속 상태:200
5페이지 확인
접속 상태:200
6페이지 확인
접속 상태:200
7페이지 확인
접속 상태:200
8페이지 확인
접속 상태:200
9페이지 확인
접속 상태:200
10페이지 확인


In [ ]:
import pandas as pd

pd.DataFrame(result).to_csv('hollys.csv',index=False,encoding='cp949')
print(result[1])

{'지역': '충북 음성군', '매장명': '국립소방병원점', '현황': '영업중', '주소': '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '매장 서비스': ['주차'], '전화번호': '042-882-0240'}


### 2. Aladin

In [ ]:
from bs4 import BeautifulSoup
import requests

URL='https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1'
headers={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0'
}

In [ ]:
# fetch,parse

# soup.select('div.ss_book_box')
def fetch(page):
    params={
        'page':page
    } # 아래 순위 버튼 눌렀을 때 생기는 파라미터 확인
    res=requests.get(URL,params=params,headers=headers,timeout=10)
    print(f'접속 상태:{res.status_code}')
    return res.text

# box.select_one('항목').text.strip()
def picker(tag,selector):
    a=tag.select_one(selector)
    return a.text.strip() if a else ''


def parse(html):
    soup=BeautifulSoup(html,'html.parser')
    boxes=soup.select('div.ss_book_box')
    rows=[]
    for box in boxes:
        # left_cover와 front_cover 두 이미지로 나뉘어있음
        # left_img=box.select_one('img.left_cover')
        # front_img=box.select_one('img.front_cover')
        imgs = box.select("img")
        image_urls = [img.get("src", "") for img in imgs]    # 이미지 여러개 수집
    
        rows.append({
            '카테고리':picker(box,'span.tit_category'),
            '제목':picker(box,'a.bo3'),
            '저자':picker(box,'li:nth-of-type(3)'),
            # Claude 왈 nth-of-type(N) — 
            # 같은 부모 안에서, 지정한 태그(li)와 같은 태그를 가진 형제 요소들만 따로 순서를 매긴 뒤 N번째 요소를 선택합니다. 
            # 이때 순서는 1부터 시작합니다(0이 아님).
            '정가':picker(box,'li:nth-of-type(4)>span'),
            '할인가':picker(box,'span.ss_p2'),
            # '이미지':front_img.get('src','') if front_img else '',
            '이미지':image_urls,
        })
    return rows

In [ ]:
result=[]
for i in range(1,11): # i:페이지
    rows=parse(fetch(i))
    result.extend(rows)
    print(f'{1+(50*(i-1))}위~{i*50}위 확인')


접속 상태:200
1위~50위 확인
접속 상태:200
51위~100위 확인
접속 상태:200
101위~150위 확인
접속 상태:200
151위~200위 확인
접속 상태:200
201위~250위 확인
접속 상태:200
251위~300위 확인
접속 상태:200
301위~350위 확인
접속 상태:200
351위~400위 확인
접속 상태:200
401위~450위 확인
접속 상태:200
451위~500위 확인


In [ ]:
import pandas as pd

pd.DataFrame(result).to_csv('aladin_bestseller.csv',index=False,encoding='cp949')
print(result[1])

{'카테고리': '[국내도서]', '제목': '그랬다고 적었다', '저자': '김애란 (지은이) | 문학동네 | 2026년 8월', '정가': '17,000', '할인가': '15,300원', '이미지': ['https://image.aladin.co.kr/product/40019/36/SpineShelf/K742130236_d.jpg', 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg']}
